In [1]:
%pip install "pandas<3.0.0" "google-genai" "python-dotenv" "openai" "scikit-learn" "seaborn" -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 822.5/822.5 kB 28.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.6 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.33.0
    Uninstalling openai-2.33.0:
      Successfully uninstalled openai-2.33.0
  Attempting uninstall: google-genai━━━━━━━━━━━━ 0/2 [openai]
    Found existing installation: google-genai 1.74.02m0/2 [openai]
    Uninstalling google-genai-1.74.0:━━━━━━━ 0/2 [openai]
      Successfully uninstalled google-genai-1.74.00/2 [openai]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [google-genai] [google-genai]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv
# loading variables from .env file
load_dotenv() 

# It will look for an environment variable named OPENAI_API_KEY
client = OpenAI()

In [2]:
import pandas as pd
from pathlib import Path

# 1. Load your engineered features dataset
features_path = '/home/liaojd/SenticCrystal/scripts/MELD/opensmile/features/egemaps_features_speaker_normalized.csv'
df_features = pd.read_csv(features_path)

# FIX: Deduplicate acoustic features to prevent many-to-many merge multiplication
initial_feat_len = len(df_features)
df_features = df_features.drop_duplicates(subset=['filename'])
print(f"Acoustic features deduplicated: {initial_feat_len} -> {len(df_features)} rows")

# 2. Load the official MELD text metadata CSVs
data_dir = Path('/home/liaojd/SenticCrystal/data/meld_7way_data') 
train_text = pd.read_csv(data_dir / 'train_sent_emo.csv')
dev_text = pd.read_csv(data_dir / 'dev_sent_emo.csv')
test_text = pd.read_csv(data_dir / 'test_sent_emo.csv')

# Add a split column to match your feature tracking paths
train_text['split'] = 'train'
dev_text['split'] = 'dev'
test_text['split'] = 'test'

# Combine all text frames together
df_text_all = pd.concat([train_text, dev_text, test_text], ignore_index=True)

# 3. Reconstruct the 'filename' to match openSMILE's tracking keys exactly
df_text_all['filename'] = (
    'dia' + df_text_all['Dialogue_ID'].astype(str) + 
    '_utt' + df_text_all['Utterance_ID'].astype(str)
)

# FIX: Keep 'Emotion' alongside 'filename' and 'Utterance'
metadata_df = df_text_all[['filename', 'Utterance', 'Emotion']].drop_duplicates(subset=['filename'])

# 4. MERGE NATIVELY
df = pd.merge(df_features, metadata_df, on='filename', how='inner')

print(f"Merge successful! Synced {len(df)} feature tracks with true text transcripts cleanly.")

# 4. MERGE NATIVELY (Strict 1-to-1 matching inner join)
df = pd.merge(df_features, metadata_df, on='filename', how='inner')

print(f"Merge successful! Synced {len(df)} feature tracks with true text transcripts cleanly.")

Acoustic features deduplicated: 13363 -> 10942 rows
Merge successful! Synced 10812 feature tracks with true text transcripts cleanly.
Merge successful! Synced 10812 feature tracks with true text transcripts cleanly.


In [ ]:
import os
import pandas as pd
from pathlib import Path
from openai import OpenAI
from tqdm import tqdm
from dotenv import load_dotenv

# 1. Initialize environment and OpenAI client
load_dotenv() 
client = OpenAI()

# 2. Setup Data Directories & Paths
base_dir = Path('/home/liaojd/SenticCrystal/scripts/MELD/opensmile')
features_path = base_dir / 'features' / 'egemaps_features_speaker_normalized.csv'
data_dir = Path('/home/liaojd/SenticCrystal/data/meld_7way_data') 

print("Loading datasets for synchronization...")
df_features = pd.read_csv(features_path)

# Deduplicate acoustic features to prevent duplicate rows
df_features = df_features.drop_duplicates(subset=['filename'])

# 3. Load official MELD text metadata
train_text = pd.read_csv(data_dir / 'train_sent_emo.csv')
dev_text = pd.read_csv(data_dir / 'dev_sent_emo.csv')
test_text = pd.read_csv(data_dir / 'test_sent_emo.csv')

train_text['split'] = 'train'
dev_text['split'] = 'dev'
test_text['split'] = 'test'

df_text_all = pd.concat([train_text, dev_text, test_text], ignore_index=True)

# Reconstruct filename matching keys (diaX_uttY)
df_text_all['filename'] = (
    'dia' + df_text_all['Dialogue_ID'].astype(str) + 
    '_utt' + df_text_all['Utterance_ID'].astype(str)
)

# FIX: Explicitly extract 'Emotion' alongside 'filename' and 'Utterance'
metadata_df = df_text_all[['filename', 'Utterance', 'Emotion']].drop_duplicates(subset=['filename'])

# 4. Strict 1-to-1 merge to inject transcripts and target labels back into features
df = pd.merge(df_features, metadata_df, on='filename', how='inner')
if 'Emotion_x' in df.columns:
    df = df.drop(columns=['Emotion_x'])
df = df.rename(columns={'Emotion_y': 'Emotion'})
print(f"Sync complete! Retained {len(df)} matching rows with valid transcripts.")

# 5. Filter for the test split block
test_df = df[df['split'] == 'test'].copy()

# Drop rows missing the generated acoustic text blocks if any exist
test_df = test_df.dropna(subset=['acoustic_prompt_injection'])
# print(list(test_df.columns))

if test_df.empty:
    raise ValueError("The test dataset slice is empty! Double-check your feature tracking filters.")

# --- 6. Prepare Full Test Set (No Sampling) ---
# Ensure your test data is sorted chronologically by conversation flow
test_df = test_df.sort_values(['Dialogue_ID', 'Utterance_ID'])

output_csv = "final_multimodal_hierarchical_results.csv"

# Checkpoint recovery: Skip dialogues that are already processed and saved
processed_ids = set()
if os.path.exists(output_csv):
    existing_df = pd.read_csv(output_csv)
    if not existing_df.empty and 'Dialogue_ID' in existing_df.columns:
        processed_ids = set(existing_df['Dialogue_ID'].unique())
        print(f"Resuming pipeline. Skipping {len(processed_ids)} already processed dialogues.")

grouped = test_df.groupby('Dialogue_ID')
results = []

print(f"Starting Full Dialogue Evaluation on {len(grouped) - len(processed_ids)} remaining conversation blocks...")

# --- 7. Optimized Hierarchical Evaluation Loop ---
for diag_id, group in tqdm(grouped, desc="Evaluating Dialogues"):
    if diag_id in processed_ids:
        continue
        
    # Convert this specific dialogue group to a list of dicts
    group_list = group.to_dict('records')
    
    # Construct the sequential conversation history block using YOUR exact prompt variables
    conversation_history = ""
    for row in group_list:
        conversation_history += (
            f"Utterance_ID: {row['Utterance_ID']}\n"
            f"Speaker: {row['Speaker']}\n"
            f"Dialogue Transcript: \"{row['Utterance']}\"\n\n"
            f"{row['acoustic_prompt_injection']}\n"
            f"----------------------------------------\n"
        )
    
    # We force the model to output JSON mapping Utterance_ID -> Emotion to completely prevent index mismatch bugs
    system_prompt = (
        "You are a multimodal emotion recognition engine. You will be given a chronological sequence of "
        "spoken lines alongside their speaker identities and voice feature acoustics.\n\n"
        "Analyze both the text and the acoustic deviations. Predict the single most likely emotion for "
        "EACH individual line in the sequence. Choose strictly from: [neutral, joy, sadness, anger, fear, surprise, disgust].\n\n"
        "Constraint: You must respond ONLY with a raw JSON object mapping the string Utterance_ID to its predicted emotion string. "
        "Do not include markdown wrappers, explanations, or text formatting. Example:\n"
        '{"0": "neutral", "1": "anger"}'
    )
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Analyze this dialogue sequence (ID {diag_id}):\n\n{conversation_history}"}
            ],
            temperature=0.0
        )
        
        # Clean up any potential markdown backticks from the response string
        raw_content = response.choices[0].message.content.strip().replace("```json", "").replace("
```", "")
        predictions_map = json.loads(raw_content)
        
        curr_batch = []
        for row in group_list:
            utt_id_str = str(row['Utterance_ID'])
            
            # Extract prediction safely; fallback to neutral if the LLM skipped an ID
            pred = predictions_map.get(utt_id_str, "neutral").strip().lower()
            
            curr_batch.append({
                'Dialogue_ID': diag_id,
                'Utterance_ID': row['Utterance_ID'],
                'Speaker': row['Speaker'],
                'Transcript': row['Utterance'],
                'Ground_Truth': row['Emotion'].lower(),
                'Predicted': pred
            })
            
        # Write batch to disk immediately. If your system disconnects, you don't lose progress or credits
        new_df = pd.DataFrame(curr_batch)
        if not os.path.isfile(output_csv):
            new_df.to_csv(output_csv, index=False)
        else:
            new_df.to_csv(output_csv, mode='a', header=False, index=False)
            
        results.extend(curr_batch)
        
    except Exception as e:
        if "429" in str(e) or "quota" in str(e).lower():
            print("\nQuota or billing balance limits reached! Saving progress and exiting safely...")
            break
        print(f"Skipping Dialogue {diag_id} due to an unexpected parsing error: {e}")

# --- 8. Complete Multi-Class Performance Analysis ---
# Load all accumulated historical results for the final analysis
if os.path.exists(output_csv):
    results_df = pd.read_csv(output_csv)
    
    # Normalize structural text outliers back to a neutral baseline
    valid_classes = ['neutral', 'joy', 'sadness', 'anger', 'fear', 'surprise', 'disgust']
    results_df['Predicted'] = results_df['Predicted'].apply(lambda x: x if str(x).lower() in valid_classes else 'neutral')
    
    results_df['Correct'] = results_df['Ground_Truth'].str.lower() == results_df['Predicted'].str.lower()
    total_accuracy = results_df['Correct'].mean() * 100

    print("\n" + "="*60)
    print(f"  FULL SYSTEM EVALUATION COMPLETE — Accuracy: {total_accuracy:.2f}%")
    print("="*60 + "\n")
    
    # Generate your Precision, Recall, and F1-Scores per emotion category
    from sklearn.metrics import classification_report
    print(classification_report(
        y_true=results_df['Ground_Truth'].str.lower(),
        y_pred=results_df['Predicted'].str.lower(),
        zero_division=0
    ))
else:
    print("\nNo evaluation rows were recorded.")

Loading datasets for synchronization...
Sync complete! Retained 10812 matching rows with valid transcripts.
Successfully balanced sample_df! Total testing rows: 35
Starting LLM Evaluation on 35 utterances...


100%|██████████| 35/35 [00:30<00:00,  1.16it/s]


Evaluation Complete! Accuracy: 51.4%

Sample Classifications:
    Speaker Ground_Truth Predicted  Correct
0    Janice        anger     anger     True
1    Monica        anger   neutral    False
2    Rachel        anger   neutral    False
3    Monica        anger     anger     True
4      Joey        anger       joy    False
5    Monica      disgust     anger    False
6    Rachel      disgust   disgust     True
7      Ross      disgust   neutral    False
8      Ross      disgust   disgust     True
9  Chandler      disgust  surprise    False


In [21]:
import os
import json
import pandas as pd
from pathlib import Path
from openai import OpenAI
from tqdm import tqdm
from dotenv import load_dotenv
from sklearn.metrics import classification_report

# 1. Initialize environment and OpenAI client
load_dotenv() 
client = OpenAI()

# 2. Setup Data Directories & Paths
base_dir = Path('/home/liaojd/SenticCrystal/scripts/MELD/opensmile')
features_path = base_dir / 'features' / 'egemaps_features_speaker_normalized.csv'
data_dir = Path('/home/liaojd/SenticCrystal/data/meld_7way_data') 

print("Loading datasets for synchronization...")
df_features = pd.read_csv(features_path)

# Deduplicate acoustic features to prevent duplicate rows
df_features = df_features.drop_duplicates(subset=['filename'])

# 3. Load official MELD text metadata
train_text = pd.read_csv(data_dir / 'train_sent_emo.csv')
dev_text = pd.read_csv(data_dir / 'dev_sent_emo.csv')
test_text = pd.read_csv(data_dir / 'test_sent_emo.csv')

train_text['split'] = 'train'
dev_text['split'] = 'dev'
test_text['split'] = 'test'

df_text_all = pd.concat([train_text, dev_text, test_text], ignore_index=True)

# Reconstruct filename matching keys (diaX_uttY)
df_text_all['filename'] = (
    'dia' + df_text_all['Dialogue_ID'].astype(str) + 
    '_utt' + df_text_all['Utterance_ID'].astype(str)
)

# Explicitly extract matching tracking columns alongside 'filename', 'Utterance', and 'Emotion'
metadata_df = df_text_all[['filename', 'Utterance', 'Emotion', 'Dialogue_ID', 'Utterance_ID', 'Speaker']].drop_duplicates(subset=['filename'])

# 4. Strict 1-to-1 merge to inject transcripts and target labels back into features
df = pd.merge(df_features, metadata_df, on='filename', how='inner')
if 'Emotion_x' in df.columns:
    df = df.drop(columns=['Emotion_x'])
df = df.rename(columns={'Emotion_y': 'Emotion'})
if 'Utterance_ID_x' in df.columns:
    df = df.drop(columns=['Utterance_ID_x'])
df = df.rename(columns={'Utterance_ID_y': 'Utterance_ID'})
if 'Dialogue_ID_x' in df.columns:
    df = df.drop(columns=['Dialogue_ID_x'])
df = df.rename(columns={'Dialogue_ID_y': 'Dialogue_ID'})
if 'Speaker_x' in df.columns:
    df = df.drop(columns=['Speaker_x'])
df = df.rename(columns={'Speaker_y': 'Speaker'})
print(f"Sync complete! Retained {len(df)} matching rows with valid transcripts.")

# 5. Filter for the test split block
test_df = df[df['split'] == 'test'].copy()

# Drop rows missing the generated acoustic text blocks if any exist
test_df = test_df.dropna(subset=['acoustic_prompt_injection'])

if test_df.empty:
    raise ValueError("The test dataset slice is empty! Double-check your feature tracking filters.")


# --- HIERARCHICAL CORE FUNCTIONS ---

def predict_dialogue_emotions_multimodal(utterances_list, diag_id):
    """
    Feeds an entire dialogue sequence to the LLM simultaneously, 
    injecting acoustic feature blocks directly underneath each line.
    Returns a dictionary mapping Utterance_ID strings to emotion labels.
    """
    # 1. Build the chronological conversation thread string with acoustic summaries
    conversation_history = ""
    for u in utterances_list:
        conversation_history += f"Utterance_ID: {u['Utterance_ID']}\n"
        conversation_history += f"Speaker: {u['Speaker']}\n"
        conversation_history += f"Line: \"{u['Utterance']}\"\n"
        conversation_history += f"Acoustic Context: {u['acoustic_prompt_injection']}\n"
        conversation_history += "----------------------------------------\n"
    
    # 2. Package explicit architectural instructions for JSON mapping
    system_prompt = (
        "You are an expert multimodal emotion recognition engine. You will be given a chronological "
        "sequence of spoken lines from a dialogue script. Underneath each line, you are provided with "
        "an 'Acoustic Context' summary representing the speaker's vocal delivery deviations (e.g., loudness, pitch changes).\n\n"
        "Tasks:\n"
        "1. Analyze the context of the conversation, structural interactions, and how the vocal acoustic summaries shift.\n"
        "2. Predict the single most likely emotion for EACH individual line in the sequence.\n"
        "3. Choose strictly from these 7 labels: [neutral, joy, sadness, anger, fear, surprise, disgust].\n\n"
        "Constraint: Respond with ONLY a raw JSON object mapping the string Utterance_ID to its predicted lowercased emotion string. "
        "Do not provide any conversational introductions, markdown formatting block wrappers, or text explanations. Example format:\n"
        '{"0": "neutral", "1": "anger"}'
    )
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Evaluate dialogue sequence ID {diag_id}:\n\n{conversation_history}"}
            ],
            temperature=0.0
        )
        
        # Clean out any accidental markdown fence blockers if the LLM adds them
        raw_content = response.choices[0].message.content.strip().replace("```json", "").replace("```", "")
        predictions_map = json.loads(raw_content)
        return predictions_map

    except Exception as e:
        print(f"API or Parsing Error during dialogue execution: {e}")
        # Fallback dictionary comprehension to maintain data safety boundaries
        return {str(u['Utterance_ID']): "neutral" for u in utterances_list}


def process_multimodal_hierarchical(test_dataset, output_csv="meld_multimodal_hierarchical_eval.csv"):
    """
    Main orchestration loop. Iterates dialogue-by-dialogue over the provided dataframe,
    handles recovery checkpoints, and prints a final classification score matrix.
    """
    # Ensure dialogues are ordered sequentially
    test_dataset = test_dataset.sort_values(['Dialogue_ID', 'Utterance_ID'])
    print(list(test_dataset.columns))
    
    # Checkpoint recovery: Skip dialogues that are already saved to the csv file
    processed_ids = set()
    if os.path.exists(output_csv):
        existing_df = pd.read_csv(output_csv)
        if not existing_df.empty and 'Dialogue_ID' in existing_df.columns:
            processed_ids = set(existing_df['Dialogue_ID'].unique())
            print(f"Resuming pipeline. Skipping {len(processed_ids)} already processed dialogues.")
    
    grouped = test_dataset.groupby('Dialogue_ID')
    results = []
    
    # Safety Valve: Set this to a small amount (e.g., 20) for testing out the architecture.
    # Change this limit or remove the check when you are ready to evaluate the full test set.
    # max_dialogues_to_test = 20 
    dialogues_counted = 0

    for diag_id, group in tqdm(grouped, desc="Processing Dialogues Hierarchically"):
        if diag_id in processed_ids:
            continue
            
        # if dialogues_counted >= max_dialogues_to_test:
        #     print(f"\nReached test limit of {max_dialogues_to_test} dialogues. Breaking to view stats.")
        #     break
            
        group_list = group.to_dict('records')
        
        try:
            # Generate predictions map via JSON keys
            predictions_map = predict_dialogue_emotions_multimodal(group_list, diag_id)
            dialogues_counted += 1
            
            curr_batch = []
            for row in group_list:
                utt_id_str = str(row['Utterance_ID'])
                # Extract matching key prediction, fallback safely to neutral
                pred = predictions_map.get(utt_id_str, "neutral")
                
                curr_batch.append({
                    'Dialogue_ID': diag_id,
                    'Utterance_ID': row['Utterance_ID'],
                    'Speaker': row['Speaker'],
                    'Ground_Truth': row['Emotion'].lower(),
                    'Predicted': str(pred).strip().lower()
                })
            
            # Save individual dialogue batch to disk instantly to protect your progress and API funds
            new_df = pd.DataFrame(curr_batch)
            if not os.path.isfile(output_csv):
                new_df.to_csv(output_csv, index=False)
            else:
                new_df.to_csv(output_csv, mode='a', header=False, index=False)
                
            results.extend(curr_batch)
            
        except Exception as e:
            if "429" in str(e) or "quota" in str(e).lower():
                print("\nQuota/Billing limit hit! Saving and exiting safely...")
                break
            print(f"Skipping Dialogue {diag_id} due to processing exception: {e}")
            
    # Load all gathered historical metrics from the file to generate the final stats report
    if os.path.exists(output_csv):
        results_df = pd.read_csv(output_csv)
        valid_classes = ['neutral', 'joy', 'sadness', 'anger', 'fear', 'surprise', 'disgust']
        
        # Normalize outliers back to a clean neutral label
        results_df['Predicted'] = results_df['Predicted'].apply(lambda x: x if str(x).lower() in valid_classes else 'neutral')
        
        results_df['Correct'] = results_df['Ground_Truth'].str.lower() == results_df['Predicted'].str.lower()
        accuracy = results_df['Correct'].mean() * 100

        print("\n=======================================================")
        print(f"   Hierarchical Multimodal Performance (Acoustic Injections)")
        print(f"   Total Dialogues Evaluated: {results_df['Dialogue_ID'].nunique()} | Full Token Accuracy: {accuracy:.1f}%")
        print("=======================================================\n")
        
        print(classification_report(
            y_true=results_df['Ground_Truth'].str.lower(),
            y_pred=results_df['Predicted'].str.lower(),
            zero_division=0
        ))
    else:
        print("\nNo rows successfully evaluated.")
        
    return pd.DataFrame(results)

if __name__ == "__main__":
    # Execute the hierarchical execution runner by passing your synced dataframe!
    final_results = process_multimodal_hierarchical(test_df, "meld_multimodal_hierarchical_eval.csv")

Loading datasets for synchronization...
Sync complete! Retained 10812 matching rows with valid transcripts.
['F0semitoneFrom27.5Hz_sma3nz_amean', 'F0semitoneFrom27.5Hz_sma3nz_stddevNorm', 'F0semitoneFrom27.5Hz_sma3nz_percentile20.0', 'F0semitoneFrom27.5Hz_sma3nz_percentile50.0', 'F0semitoneFrom27.5Hz_sma3nz_percentile80.0', 'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2', 'F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope', 'F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope', 'F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope', 'F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope', 'loudness_sma3_amean', 'loudness_sma3_stddevNorm', 'loudness_sma3_percentile20.0', 'loudness_sma3_percentile50.0', 'loudness_sma3_percentile80.0', 'loudness_sma3_pctlrange0-2', 'loudness_sma3_meanRisingSlope', 'loudness_sma3_stddevRisingSlope', 'loudness_sma3_meanFallingSlope', 'loudness_sma3_stddevFallingSlope', 'spectralFlux_sma3_amean', 'spectralFlux_sma3_stddevNorm', 'mfcc1_sma3_amean', 'mfcc1_sma3_stddevNorm', 'mfcc2_sma3_amean

Processing Dialogues Hierarchically: 100%|██████████| 249/249 [05:08<00:00,  1.24s/it]


   Hierarchical Multimodal Performance (Acoustic Injections)
   Total Dialogues Evaluated: 249 | Full Token Accuracy: 61.5%

              precision    recall  f1-score   support

       anger       0.54      0.52      0.53       173
     disgust       0.19      0.26      0.22        34
        fear       0.25      0.34      0.29        38
         joy       0.53      0.71      0.60       258
     neutral       0.80      0.67      0.73       727
     sadness       0.34      0.44      0.39        90
    surprise       0.63      0.54      0.58       157

    accuracy                           0.62      1477
   macro avg       0.47      0.50      0.48      1477
weighted avg       0.65      0.62      0.63      1477

